In [24]:

from pathlib import Path

import plotly.express as px
import plotly.express as px
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import numpy as np
import plotly.graph_objects as go
from statsmodels.nonparametric.smoothers_lowess import lowess
import pandas as pd

from IPython.display import HTML

In [2]:
# DATA DIRECTORIES
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'data_cleaned'
IMG_DIR = DATA_DIR / 'images'
DOC_THEMES_DIR = DATA_DIR / 'THEMES_RESULTS_V4'/'docs_with_themes.csv'
THEMES_TFIDF_DIR=DATA_DIR/'THEMES_RESULTS_V4'/'TF_IDF_themes_scores.csv'
THEMES_DIR=DATA_DIR/'THEMES_RESULTS_V4'/'themes.csv'
Themes_aug_DIR = DATA_DIR / 'Themes_augmented_3.csv'
OLLAMA_LABELS_DIR = DATA_DIR / 'themes_ollama.csv'

In [14]:
themes_df = pd.read_csv(OLLAMA_LABELS_DIR)
theme_summary = themes_df[[
    "llm_theme_label",
    "nb_instances",
    "mu_mean"
]].dropna()
theme_summary = theme_summary.rename(columns={
    "nb_instances": "frequency",
    "mu_mean": "mean_funniness"
})
print(len(themes_df))

80


In [ ]:

fig = px.scatter(
    themes_df,
    x="nb_instances",
    y="mu_mean",
    hover_data=["theme_id", "llm_theme_label"],
    labels={
        "nb_instances": "Number of captions (frequency)",
        "mu_mean": "Mean of captions (μ)",
        "llm_theme_label": "Caption theme",
    },
    title="Themes funiness vs frequency with LOWESS trendline",
    color="llm_theme_label",
)

fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color="black")))

lowess_line = lowess(
    endog=themes_df["mu_mean"],
    exog=themes_df["nb_instances"],
    frac=0.25,  # smoothness; tweak 0.2–0.5 as needed
    it=0,
    return_sorted=True,
)

fig.add_trace(
    go.Scatter(
        x=lowess_line[:, 0],
        y=lowess_line[:, 1],
        mode="lines",
        line=dict(color="crimson", width=3),
        name="LOWESS trend",
        hoverinfo="skip",
    )
)

fig.show()


In [25]:
rho, p = spearmanr(themes_df["nb_instances"], themes_df["mu_mean"])

print(f"Spearman rho: {rho:.3f}")
print(f"p-value: {p:.3g}")

Spearman rho: -0.048
p-value: 0.674
